In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('divar.csv', low_memory=False)
df_sale = df[df['price_value'].notna()].copy()

def build_train_val_test(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    return X_train.copy(), X_val.copy(), X_test.copy(), y_train, y_val, y_test

def clean_pipeline(X_train, X_val, X_test):
    drop_cols = ['description', 'title', 'id', 'token']

    X_train = X_train.drop(columns=drop_cols, errors='ignore')
    X_val = X_val.drop(columns=drop_cols, errors='ignore')
    X_test = X_test.drop(columns=drop_cols, errors='ignore')

    all_nan_cols = X_train.columns[X_train.isna().all()].tolist()

    X_train = X_train.drop(columns=all_nan_cols)
    X_val = X_val.drop(columns=all_nan_cols, errors='ignore')
    X_test = X_test.drop(columns=all_nan_cols, errors='ignore')

    numeric_like_cols = ['floor', 'rooms_count', 'construction_year']

    for col in numeric_like_cols:
        if col in X_train.columns:
            X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
            X_val[col] = pd.to_numeric(X_val[col], errors='coerce')
            X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

    bool_cols = ['has_balcony', 'has_warm_water_provider', 'has_restroom', 'is_rebuilt']
    bool_map = {'true': 1, 'false': 0, True: 1, False: 0}

    for col in bool_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].map(bool_map)
            X_val[col] = X_val[col].map(bool_map)
            X_test[col] = X_test[col].map(bool_map)

    if 'created_at_month' in X_train.columns:
        for d in (X_train, X_val, X_test):
            d['created_at_month'] = pd.to_datetime(d['created_at_month'],errors='coerce')
            month = d['created_at_month'].dt.month
            d['month_sin'] = np.sin(2 * np.pi * month / 12)
            d['month_cos'] = np.cos(2 * np.pi * month / 12)
            d.drop(columns=['created_at_month'], inplace=True)

    missing_ratio = X_train.isna().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.6].index

    X_train = X_train.drop(columns=cols_to_drop)
    X_val = X_val.drop(columns=cols_to_drop, errors='ignore')
    X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

    numeric_cols = X_train.select_dtypes(include='number').columns
    categorical_cols = X_train.select_dtypes(exclude='number').columns

    for col in numeric_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

    for col in categorical_cols:
        mode_val = X_train[col].mode()[0]
        X_train[col] = X_train[col].fillna(mode_val)
        X_val[col] = X_val[col].fillna(mode_val)
        X_test[col] = X_test[col].fillna(mode_val)

    return X_train, X_val, X_test

def get_percentile_bounds(series, lower_pct=0.01, upper_pct=0.99):
    return series.quantile(lower_pct), series.quantile(upper_pct)

def outlier_pipeline(X_train, X_val, X_test, cols):
    for col in cols:
        if col not in X_train.columns:
            continue
        lower, upper = get_percentile_bounds(X_train[col])
        X_train[col] = X_train[col].clip(lower, upper)
        X_val[col] = X_val[col].clip(lower, upper)
        X_test[col] = X_test[col].clip(lower, upper)

    return X_train, X_val, X_test

def encode_and_scale_pipeline(X_train, X_val, X_test):
    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()

    X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_val = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    numeric_cols = [col for col in numeric_cols if col in X_train.columns]
    scaler = StandardScaler()

    X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val.loc[:, numeric_cols] = scaler.transform(X_val[numeric_cols])
    X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

    return X_train, X_val, X_test

def remove_target_outliers(X, y, lower_pct=0.01, upper_pct=0.98):
    lower_bound = y.quantile(lower_pct)
    upper_bound = y.quantile(upper_pct)
    mask = (y >= lower_bound) & (y <= upper_bound)
    return X[mask].copy(), y[mask].copy()


X_train_sale, X_val_sale, X_test_sale, y_train_sale, y_val_sale, y_test_sale = build_train_val_test(df_sale, 'price_value')

X_train_sale, y_train_sale = remove_target_outliers(X_train_sale, y_train_sale)
X_val_sale, y_val_sale = remove_target_outliers(X_val_sale, y_val_sale)
X_test_sale, y_test_sale = remove_target_outliers(X_test_sale, y_test_sale)

y_train_sale_log = np.log1p(y_train_sale)
y_val_sale_log = np.log1p(y_val_sale)
y_test_sale_log = np.log1p(y_test_sale)

X_train_sale, X_val_sale, X_test_sale = clean_pipeline(X_train_sale, X_val_sale, X_test_sale)
outlier_cols = ['building_size', 'land_size', 'floor', 'rooms_count', 'construction_year', 'location_radius']
X_train_sale, X_val_sale, X_test_sale = outlier_pipeline(X_train_sale, X_val_sale, X_test_sale, outlier_cols)
X_train_sale, X_val_sale, X_test_sale = encode_and_scale_pipeline(X_train_sale, X_val_sale, X_test_sale)

/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_5971/2444908416.py:74: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].fillna(mode_val)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_5971/2444908416.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val[col] = X_val[col].fillna(mode_val)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_5971/2444908416.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result

In [2]:
from skopt import BayesSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

search_spaces = {
    'learning_rate': (0.02, 0.2, 'log-uniform'),
    'max_depth': (3, 8),
    'min_child_weight': (1, 8),
    'subsample': (0.6, 1.0, 'uniform'),
    'colsample_bytree': (0.6, 1.0, 'uniform'),
    'gamma': (0.0, 5.0, 'uniform'),
    'reg_alpha': (1e-4, 10.0, 'log-uniform'),
    'reg_lambda': (1e-3, 10.0, 'log-uniform'),
    'n_estimators': (150, 500)
}

model = XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    random_state=42,
    n_jobs=1
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

bayes_search = BayesSearchCV(
    estimator=model,
    search_spaces=search_spaces,
    scoring='r2',
    cv=cv,
    n_iter=30,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

bayes_search.fit(X_train_sale, y_train_sale)

best_model = bayes_search.best_estimator_

y_test_pred = best_model.predict(X_test_sale)

test_mae = mean_absolute_error(y_test_sale, y_test_pred)
test_mse = mean_squared_error(y_test_sale, y_test_pred)
test_r2 = r2_score(y_test_sale, y_test_pred)

print("Best Parameters:")
print(bayes_search.best_params_)

print(f"Best CV R2: {bayes_search.best_score_:.4f}")
print(f"MAE: {test_mae:.4f}")
print(f"MSE: {test_mse:.4f}")
print(f"R2 : {test_r2:.4f}")

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi